# Coffee Sales Data Analysis
## Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on the Coffee Shop Sales dataset.

### Contents:
1. Data Loading
2. Data Overview
3. Data Cleaning & Preprocessing
4. Statistical Analysis
5. Visualization
6. Insights & Conclusions

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Data Loading

In [ ]:
# Load the data
data_path = Path('../data/cleaned/')

# Try to load parquet first, then CSV
parquet_files = list(data_path.glob('*.parquet'))
if parquet_files:
    latest_file = max(parquet_files, key=lambda x: x.stat().st_mtime)
    df = pd.read_parquet(latest_file)
    print(f"Loaded data from: {latest_file.name}")
else:
    csv_files = list(data_path.glob('*.csv'))
    latest_file = max(csv_files, key=lambda x: x.stat().st_mtime)
    df = pd.read_csv(latest_file)
    print(f"Loaded data from: {latest_file.name}")

print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

## 2. Data Overview

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Data types and info
print("Dataset Information:")
df.info()

In [ ]:
# Basic statistics
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing_Count'] > 0])

## 3. Data Cleaning & Preprocessing

In [ ]:
# Parse date columns if not already datetime
date_columns = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]

for col in date_columns:
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"Converted {col} to datetime")

## 4. Statistical Analysis

In [ ]:
# Correlation matrix for numeric columns
numeric_df = df.select_dtypes(include=[np.number])

if len(numeric_df.columns) > 1:
    plt.figure(figsize=(12, 8))
    correlation_matrix = numeric_df.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Correlation Matrix of Numeric Features')
    plt.tight_layout()
    plt.show()

## 5. Visualization

In [ ]:
# Distribution of numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

if len(numeric_cols) > 0:
    fig, axes = plt.subplots(nrows=len(numeric_cols), ncols=2, figsize=(15, 5*len(numeric_cols)))
    if len(numeric_cols) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, col in enumerate(numeric_cols):
        # Histogram
        axes[idx, 0].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
        axes[idx, 0].set_title(f'Distribution of {col}')
        axes[idx, 0].set_xlabel(col)
        axes[idx, 0].set_ylabel('Frequency')
        
        # Box plot
        axes[idx, 1].boxplot(df[col].dropna(), vert=True)
        axes[idx, 1].set_title(f'Box Plot of {col}')
        axes[idx, 1].set_ylabel(col)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Top categories in categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols[:5]:  # Limit to first 5 categorical columns
    print(f"\nTop 10 values in {col}:")
    top_values = df[col].value_counts().head(10)
    print(top_values)
    
    # Bar plot
    plt.figure(figsize=(10, 6))
    top_values.plot(kind='bar', color='steelblue')
    plt.title(f'Top 10 Values in {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Time series analysis (if date columns exist)
if date_columns:
    date_col = date_columns[0]
    
    # Daily counts
    daily_counts = df.groupby(df[date_col].dt.date).size()
    
    plt.figure(figsize=(15, 6))
    daily_counts.plot(kind='line', color='darkblue')
    plt.title('Daily Transaction Count Over Time')
    plt.xlabel('Date')
    plt.ylabel('Number of Transactions')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Insights & Conclusions

In [ ]:
# Summary statistics
print("Key Insights:")
print("="*50)
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

if len(numeric_cols) > 0:
    print(f"\nNumeric Columns: {len(numeric_cols)}")
    for col in numeric_cols:
        print(f"  - {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")

if len(categorical_cols) > 0:
    print(f"\nCategorical Columns: {len(categorical_cols)}")
    for col in categorical_cols:
        print(f"  - {col}: {df[col].nunique()} unique values")

## Next Steps

Based on this exploratory analysis:
1. Identify key patterns and trends
2. Develop specific business questions
3. Build predictive models
4. Create dashboards for stakeholders
5. Set up automated reporting